March Madness / college basketball prediction project centered on building a model that predicts NCAA tournament games from historical team-level data.

In [1]:
import bisect
from collections import defaultdict
import warnings

import numpy as np
import pandas as pd
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.base import clone
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore", message="X does not have valid feature names")

In [13]:
CURRENT_SEASON = 2026
DATA_DIR = 'march-machine-learning-mania-2026'
TOURNAMENT_WEIGHT = 1  # how much more to weight tournament games vs regular season

# Ordinal ranking systems with perfect 24-season coverage (2003-2026)
# Used for computing cross-system stats (std dev, min rank)
ORDINAL_SYSTEMS = ['COL', 'DOL', 'MOR', 'POM', 'WLK']

# Subset exposed as individual features (maximizes methodological diversity)
# POM = adjusted efficiency, MOR = margin + SOS (highest deviation), COL = pure win/loss
ORDINAL_INDIVIDUAL_SYSTEMS = ['POM', 'MOR', 'COL']

# Allowlist of base features to include in training
# Raw counting stats excluded — their signal is captured by rates/efficiencies
ALLOWED_FEATURES = {
    # Shooting rates (pace-independent)
    'fg_pct', 'fg3_pct', 'ft_pct', 'fg3_rate',
    'opponent_fg_pct', 'opponent_fg3_pct', 'opponent_ft_pct', 'opponent_fg3_rate',
    # Dean Oliver's Four Factors
    'offensive_efg_pct', 'defensive_efg_pct',
    'offensive_tov_pct', 'defensive_tov_pct',
    'offensive_reb_pct', 'defensive_reb_pct',
    'offensive_ft_rate', 'defensive_ft_rate',
    # Efficiency ratings (Bayesian-updated, SOS-adjusted)
    'offensive_efficiency', 'defensive_efficiency',
    # Tempo
    'avg_possessions',
    # Defensive activity (own stats — more stable than opponent versions)
    'avg_stl', 'avg_blk',
    # Rebounds (raw — may remove in Tier 2)
    'avg_or', 'avg_dr', 'avg_opponent_or', 'avg_opponent_dr',
    # Turnovers (raw — may remove in Tier 2)
    'avg_to', 'avg_opponent_to',
    # Assists (raw — may remove in Tier 2)
    'avg_ast', 'avg_opponent_ast',
}

In [3]:
csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
data = {os.path.splitext(f)[0]: pd.read_csv(os.path.join(DATA_DIR, f)) for f in sorted(csv_files)}

for name, df in data.items():
    print(f"{name}: {df.shape}")

Cities: (510, 3)
Conferences: (51, 2)
MConferenceTourneyGames: (7093, 5)
MGameCities: (92438, 6)
MMasseyOrdinals: (5865001, 5)
MNCAATourneyCompactResults: (2585, 8)
MNCAATourneyDetailedResults: (1449, 34)
MNCAATourneySeedRoundSlots: (776, 5)
MNCAATourneySeeds: (2694, 3)
MNCAATourneySlots: (2653, 4)
MRegularSeasonCompactResults: (198577, 8)
MRegularSeasonDetailedResults: (124529, 34)
MSeasons: (42, 6)
MSecondaryTourneyCompactResults: (1865, 9)
MSecondaryTourneyTeams: (1895, 3)
MTeamCoaches: (13900, 5)
MTeamConferences: (13753, 3)
MTeamSpellings: (1178, 2)
MTeams: (381, 4)
SampleSubmissionStage1: (519144, 2)
SampleSubmissionStage2: (132133, 2)
WConferenceTourneyGames: (6777, 5)
WGameCities: (89035, 6)
WNCAATourneyCompactResults: (1717, 8)
WNCAATourneyDetailedResults: (961, 34)
WNCAATourneySeeds: (1812, 3)
WNCAATourneySlots: (1847, 4)
WRegularSeasonCompactResults: (142507, 8)
WRegularSeasonDetailedResults: (87187, 34)
WSeasons: (29, 6)
WSecondaryTourneyCompactResults: (906, 9)
WSecondaryT

In [4]:
display(data['MRegularSeasonDetailedResults'])

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,...,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,...,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,...,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,...,16,17,27,21,15,12,10,7,1,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124524,2026,132,1335,88,1463,84,N,1,29,64,...,24,13,17,17,24,23,9,9,4,16
124525,2026,132,1345,80,1276,72,N,0,30,57,...,24,5,6,11,22,15,7,2,3,18
124526,2026,132,1378,70,1455,55,N,0,23,61,...,19,13,20,12,27,8,14,7,7,19
124527,2026,132,1433,70,1173,62,N,0,23,57,...,21,9,17,8,31,16,11,3,4,18


In [5]:
print(data['MRegularSeasonDetailedResults'].columns)

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR',
       'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3',
       'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'],
      dtype='str')


In [6]:
BOX_STATS = ['Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
             'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']

PRIOR_OFFENSIVE_EFFICIENCY = 100.0
PRIOR_DEFENSIVE_EFFICIENCY = 100.0
EFFICIENCY_LEARNING_RATE = 0.10
LEAGUE_PRIOR_GAMES = 200
AVERAGE_POSSESSIONS_PER_TEAM = 69.0
HOME_COURT_ADVANTAGE = 3.5  # points per 100 possessions
SEASON_CARRYOVER_WEIGHT = 0.5
RESIDUAL_CAP = 15.0

regular_season_results = data['MRegularSeasonDetailedResults'].copy()
regular_season_results['is_tournament'] = 0

tournament_results = data['MNCAATourneyDetailedResults'].copy()
tournament_results['is_tournament'] = 1

results = pd.concat([regular_season_results, tournament_results], ignore_index=True).sort_values(
    ['Season', 'DayNum']).reset_index(drop=True)

# Build tournament seed lookup: (season, team_id) -> seed number (1-16)
seed_lookup = {}
for _, seed_row in data['MNCAATourneySeeds'].iterrows():
    seed_number = int(seed_row['Seed'][1:3])  # "W01" -> 1, "X16a" -> 16
    seed_lookup[(seed_row['Season'], seed_row['TeamID'])] = seed_number

# Build ordinal rank lookup: (season, team_id) -> sorted list of (ranking_day_num, avg_rank)
# Group by (Season, RankingDayNum, TeamID), average OrdinalRank across all ranking systems
ordinal_averages = (data['MMasseyOrdinals']
    .groupby(['Season', 'RankingDayNum', 'TeamID'])['OrdinalRank']
    .mean()
    .reset_index())

ordinal_lookup = defaultdict(list)
for _, row in ordinal_averages.iterrows():
    ordinal_lookup[(row['Season'], row['TeamID'])].append(
        (row['RankingDayNum'], row['OrdinalRank']))

# Sort each list by RankingDayNum for binary search
for key in ordinal_lookup:
    ordinal_lookup[key].sort()

print(f"Ordinal lookup entries: {len(ordinal_lookup)}")

# Build per-system ordinal lookup: (system_name, season, team_id) -> sorted [(day, rank)]
selected_ordinals = data['MMasseyOrdinals'][data['MMasseyOrdinals']['SystemName'].isin(ORDINAL_SYSTEMS)]

system_ordinal_lookup = defaultdict(list)
for _, row in selected_ordinals.iterrows():
    system_ordinal_lookup[(row['SystemName'], row['Season'], row['TeamID'])].append(
        (row['RankingDayNum'], row['OrdinalRank']))

for key in system_ordinal_lookup:
    system_ordinal_lookup[key].sort()

print(f"System ordinal lookup entries: {len(system_ordinal_lookup)}")

# Build ordinal stats lookup: (season, team_id) -> sorted [(day, std_dev, min_rank)]
# Group selected systems by (Season, RankingDayNum, TeamID)
ordinal_stats_grouped = (selected_ordinals
    .groupby(['Season', 'RankingDayNum', 'TeamID'])['OrdinalRank']
    .agg(['std', 'min'])
    .reset_index())

ordinal_stats_lookup = defaultdict(list)
for _, row in ordinal_stats_grouped.iterrows():
    std_value = row['std'] if not np.isnan(row['std']) else 0.0
    ordinal_stats_lookup[(row['Season'], row['TeamID'])].append(
        (row['RankingDayNum'], std_value, row['min']))

for key in ordinal_stats_lookup:
    ordinal_stats_lookup[key].sort()

print(f"Ordinal stats lookup entries: {len(ordinal_stats_lookup)}")

def get_ordinal_rank(season, team_id, day_num):
    """Get the average ordinal rank for a team using the most recent rankings available before day_num."""
    rankings = ordinal_lookup.get((season, team_id))
    if not rankings:
        return np.nan
    # Binary search for largest RankingDayNum <= day_num
    day_nums = [r[0] for r in rankings]
    idx = bisect.bisect_right(day_nums, day_num) - 1
    if idx < 0:
        return np.nan
    return rankings[idx][1]

def get_system_ordinal_rank(system_name, season, team_id, day_num):
    """Get the ordinal rank from a specific ranking system using the most recent available before day_num."""
    rankings = system_ordinal_lookup.get((system_name, season, team_id))
    if not rankings:
        return np.nan
    day_nums = [r[0] for r in rankings]
    idx = bisect.bisect_right(day_nums, day_num) - 1
    if idx < 0:
        return np.nan
    return rankings[idx][1]

def get_ordinal_stats(season, team_id, day_num):
    """Get (std_dev, min_rank) across the 5 selected systems using the most recent available before day_num."""
    stats = ordinal_stats_lookup.get((season, team_id))
    if not stats:
        return np.nan, np.nan
    day_nums = [r[0] for r in stats]
    idx = bisect.bisect_right(day_nums, day_num) - 1
    if idx < 0:
        return np.nan, np.nan
    return stats[idx][1], stats[idx][2]

def build_ordinal_features(season, team_id, day_num):
    """Build all ordinal features for a team: mean rank, individual system ranks, std dev, min rank."""
    features = {}
    features['ordinal_rank'] = get_ordinal_rank(season, team_id, day_num)
    for system_name in ORDINAL_INDIVIDUAL_SYSTEMS:
        features[f'{system_name.lower()}_rank'] = get_system_ordinal_rank(system_name, season, team_id, day_num)
    std_dev, min_rank = get_ordinal_stats(season, team_id, day_num)
    features['ordinal_std'] = std_dev
    features['ordinal_min'] = min_rank
    return features

# {(season, team_id): {'sums': {stat: float}, 'game_count': int, ...}}
team_rolling = {}

# {season: {'total_points': float, 'total_possessions': float, 'game_count': int}}
league_totals = {}

def get_or_create_entry(season, team_id):
    key = (season, team_id)
    if key not in team_rolling:
        sums = {}
        for stat in BOX_STATS:
            sums[stat] = 0.0
            sums[f'opponent_{stat}'] = 0.0

        # Season-to-season carryover with regression to mean
        prior_key = (season - 1, team_id)
        if prior_key in team_rolling:
            prior = team_rolling[prior_key]
            initial_offensive = (SEASON_CARRYOVER_WEIGHT * prior['offensive_efficiency']
                                 + (1 - SEASON_CARRYOVER_WEIGHT) * PRIOR_OFFENSIVE_EFFICIENCY)
            initial_defensive = (SEASON_CARRYOVER_WEIGHT * prior['defensive_efficiency']
                                 + (1 - SEASON_CARRYOVER_WEIGHT) * PRIOR_DEFENSIVE_EFFICIENCY)
        else:
            initial_offensive = PRIOR_OFFENSIVE_EFFICIENCY
            initial_defensive = PRIOR_DEFENSIVE_EFFICIENCY

        team_rolling[key] = {
            'sums': sums,
            'game_count': 0,
            'offensive_efficiency': initial_offensive,
            'defensive_efficiency': initial_defensive,
            'total_possessions': 0.0,
        }
    return team_rolling[key]

def estimate_possessions(game, prefix):
    # FGA - OR + TO + 0.475 * FTA (standard Kenpom possession estimate)
    return game[f'{prefix}FGA'] - game[f'{prefix}OR'] + game[f'{prefix}TO'] + 0.475 * game[f'{prefix}FTA']

def get_or_create_league_totals(season):
    if season not in league_totals:
        league_totals[season] = {'total_points': 0.0, 'total_possessions': 0.0, 'game_count': 0}
    return league_totals[season]

def get_league_average_efficiency(season):
    totals = get_or_create_league_totals(season)

    phantom_games = max(0, LEAGUE_PRIOR_GAMES - totals['game_count'])
    if phantom_games == 0:
        return totals['total_points'] / totals['total_possessions'] * 100.0

    if season - 1 in league_totals and league_totals[season - 1]['total_possessions'] > 0:
        prior_efficiency = league_totals[season - 1]['total_points'] / league_totals[season - 1]['total_possessions'] * 100.0
    else:
        prior_efficiency = 100.0

    phantom_possessions = phantom_games * 2 * AVERAGE_POSSESSIONS_PER_TEAM
    phantom_points = phantom_possessions * prior_efficiency / 100.0

    blended_points = phantom_points + totals['total_points']
    blended_possessions = phantom_possessions + totals['total_possessions']
    return blended_points / blended_possessions * 100.0

def update_team_box_stats(entry, game, own_prefix, opp_prefix):
    entry['game_count'] += 1
    for stat in BOX_STATS:
        entry['sums'][stat] += game[f'{own_prefix}{stat}']
        entry['sums'][f'opponent_{stat}'] += game[f'{opp_prefix}{stat}']

def extract_team_features(entry):
    """Extract features from a team's rolling entry."""
    game_count = entry['game_count']
    sums = entry['sums']
    features = {}

    # 28 per-game averages (14 own + 14 opponent)
    for stat in BOX_STATS:
        features[f'avg_{stat.lower()}'] = sums[stat] / game_count
        features[f'avg_opponent_{stat.lower()}'] = sums[f'opponent_{stat}'] / game_count

    # 8 rate stats (from sums for proper weighting)
    def safe_divide(numerator, denominator):
        return numerator / denominator if denominator > 0 else 0.0

    features['fg_pct'] = safe_divide(sums['FGM'], sums['FGA'])
    features['fg3_pct'] = safe_divide(sums['FGM3'], sums['FGA3'])
    features['ft_pct'] = safe_divide(sums['FTM'], sums['FTA'])
    features['fg3_rate'] = safe_divide(sums['FGA3'], sums['FGA'])
    features['opponent_fg_pct'] = safe_divide(sums['opponent_FGM'], sums['opponent_FGA'])
    features['opponent_fg3_pct'] = safe_divide(sums['opponent_FGM3'], sums['opponent_FGA3'])
    features['opponent_ft_pct'] = safe_divide(sums['opponent_FTM'], sums['opponent_FTA'])
    features['opponent_fg3_rate'] = safe_divide(sums['opponent_FGA3'], sums['opponent_FGA'])

    # Dean Oliver's Four Factors (offense and defense)
    # 1. Effective Field Goal % — eFG% = (FG + 0.5 * 3P) / FGA
    features['offensive_efg_pct'] = safe_divide(sums['FGM'] + 0.5 * sums['FGM3'], sums['FGA'])
    features['defensive_efg_pct'] = safe_divide(sums['opponent_FGM'] + 0.5 * sums['opponent_FGM3'], sums['opponent_FGA'])

    # 2. Turnover % — TOV% = TOV / (FGA + 0.44 * FTA + TOV)
    features['offensive_tov_pct'] = safe_divide(sums['TO'], sums['FGA'] + 0.44 * sums['FTA'] + sums['TO'])
    features['defensive_tov_pct'] = safe_divide(sums['opponent_TO'], sums['opponent_FGA'] + 0.44 * sums['opponent_FTA'] + sums['opponent_TO'])

    # 3. Rebound % — ORB% = ORB / (ORB + Opp DRB), DRB% = DRB / (Opp ORB + DRB)
    features['offensive_reb_pct'] = safe_divide(sums['OR'], sums['OR'] + sums['opponent_DR'])
    features['defensive_reb_pct'] = safe_divide(sums['DR'], sums['opponent_OR'] + sums['DR'])

    # 4. Free Throw Rate — FT / FGA
    features['offensive_ft_rate'] = safe_divide(sums['FTM'], sums['FGA'])
    features['defensive_ft_rate'] = safe_divide(sums['opponent_FTM'], sums['opponent_FGA'])

    # 3 efficiency
    features['offensive_efficiency'] = entry['offensive_efficiency']
    features['defensive_efficiency'] = entry['defensive_efficiency']
    features['net_efficiency'] = entry['offensive_efficiency'] - entry['defensive_efficiency']

    # 1 tempo
    features['avg_possessions'] = entry['total_possessions'] / game_count

    return features

def featurize_game(team_a_entry, team_b_entry, season, day_number, team_a_id, team_b_id,
                   label, team_a_location, is_tournament, team_a_seed, team_b_seed,
                   team_a_ordinals, team_b_ordinals):
    """Build a feature row from two teams' rolling entries (team_a = lower ID).

    Returns None if either team has no games yet.
    """
    if team_a_entry['game_count'] == 0 or team_b_entry['game_count'] == 0:
        return None

    team_a_features = extract_team_features(team_a_entry)
    team_b_features = extract_team_features(team_b_entry)

    row = {}
    for key, value in team_a_features.items():
        row[f'team_a_{key}'] = value
    for key, value in team_b_features.items():
        row[f'team_b_{key}'] = value
    for key in team_a_features:
        row[f'diff_{key}'] = team_a_features[key] - team_b_features[key]

    row['team_a_seed'] = team_a_seed
    row['team_b_seed'] = team_b_seed

    for key, value in team_a_ordinals.items():
        row[f'team_a_{key}'] = value
    for key, value in team_b_ordinals.items():
        row[f'team_b_{key}'] = value

    row['season'] = season
    row['day_number'] = day_number
    row['team_a_id'] = team_a_id
    row['team_b_id'] = team_b_id
    row['label'] = label
    row['team_a_location'] = team_a_location
    row['is_tournament'] = is_tournament

    return row

training_rows = []

for _, game in results.iterrows():
    winner_entry = get_or_create_entry(game['Season'], game['WTeamID'])
    loser_entry = get_or_create_entry(game['Season'], game['LTeamID'])

    # Featurize both teams' rolling stats (pre-update) for training data
    winner_team_id = game['WTeamID']
    loser_team_id = game['LTeamID']

    # Look up seeds only for tournament games
    if game['is_tournament'] == 1:
        winner_seed = seed_lookup.get((game['Season'], winner_team_id), np.nan)
        loser_seed = seed_lookup.get((game['Season'], loser_team_id), np.nan)
    else:
        winner_seed = np.nan
        loser_seed = np.nan

    # Look up ordinal features for both teams
    winner_ordinals = build_ordinal_features(game['Season'], winner_team_id, game['DayNum'])
    loser_ordinals = build_ordinal_features(game['Season'], loser_team_id, game['DayNum'])

    if winner_team_id < loser_team_id:
        team_a_location = game['WLoc']
        row = featurize_game(winner_entry, loser_entry, game['Season'], game['DayNum'],
                             winner_team_id, loser_team_id, label=1, team_a_location=team_a_location,
                             is_tournament=game['is_tournament'],
                             team_a_seed=winner_seed, team_b_seed=loser_seed,
                             team_a_ordinals=winner_ordinals, team_b_ordinals=loser_ordinals)
    else:
        location_flip = {'H': 'A', 'A': 'H', 'N': 'N'}
        team_a_location = location_flip[game['WLoc']]
        row = featurize_game(loser_entry, winner_entry, game['Season'], game['DayNum'],
                             loser_team_id, winner_team_id, label=0, team_a_location=team_a_location,
                             is_tournament=game['is_tournament'],
                             team_a_seed=loser_seed, team_b_seed=winner_seed,
                             team_a_ordinals=loser_ordinals, team_b_ordinals=winner_ordinals)
    if row is not None:
        training_rows.append(row)

    # --- Adjusted efficiency update ---
    # expected = own_off + opp_def - league_avg; update ratings by k * (raw - expected)
    possessions = (estimate_possessions(game, 'W') + estimate_possessions(game, 'L')) / 2

    if possessions > 0:
        winner_raw_offensive = game['WScore'] / possessions * 100.0
        loser_raw_offensive = game['LScore'] / possessions * 100.0

        # Home court adjustment: remove HCA from raw efficiencies
        if game['WLoc'] == 'H':
            winner_raw_offensive -= HOME_COURT_ADVANTAGE
            loser_raw_offensive += HOME_COURT_ADVANTAGE
        elif game['WLoc'] == 'A':
            winner_raw_offensive += HOME_COURT_ADVANTAGE
            loser_raw_offensive -= HOME_COURT_ADVANTAGE
        # 'N': no adjustment

        league_average = get_league_average_efficiency(game['Season'])

        winner_expected_offensive = (winner_entry['offensive_efficiency']
                                     + loser_entry['defensive_efficiency']
                                     - league_average)
        loser_expected_offensive = (loser_entry['offensive_efficiency']
                                    + winner_entry['defensive_efficiency']
                                    - league_average)

        # Residual capping to limit impact of outlier games
        winner_offensive_residual = np.clip(winner_raw_offensive - winner_expected_offensive, -RESIDUAL_CAP, RESIDUAL_CAP)
        loser_offensive_residual = np.clip(loser_raw_offensive - loser_expected_offensive, -RESIDUAL_CAP, RESIDUAL_CAP)

        winner_entry['offensive_efficiency'] += EFFICIENCY_LEARNING_RATE * winner_offensive_residual
        loser_entry['defensive_efficiency'] += EFFICIENCY_LEARNING_RATE * winner_offensive_residual
        loser_entry['offensive_efficiency'] += EFFICIENCY_LEARNING_RATE * loser_offensive_residual
        winner_entry['defensive_efficiency'] += EFFICIENCY_LEARNING_RATE * loser_offensive_residual

    # Accumulate league totals
    season_totals = get_or_create_league_totals(game['Season'])
    season_totals['total_points'] += game['WScore'] + game['LScore']
    season_totals['total_possessions'] += 2 * possessions
    season_totals['game_count'] += 1

    # Accumulate team possessions
    winner_entry['total_possessions'] += possessions
    loser_entry['total_possessions'] += possessions

    update_team_box_stats(winner_entry, game, 'W', 'L')
    update_team_box_stats(loser_entry, game, 'L', 'W')

print(f"Processed {len(results)} games, {len(team_rolling)} team-seasons")
print(f"Training rows collected: {len(training_rows)}")
print(f"Seed lookup entries: {len(seed_lookup)}")

Ordinal lookup entries: 8356
System ordinal lookup entries: 41729
Ordinal stats lookup entries: 8355
Processed 125978 games, 8346 team-seasons
Training rows collected: 120861
Seed lookup entries: 2694


In [7]:
training_data = pd.DataFrame(training_rows)
print(f"Training rows: {training_data.shape[0]}, Features: {training_data.shape[1] - 6}")
print(f"Label balance: {training_data['label'].mean():.3f}")
print(f"Location values: {sorted(training_data['team_a_location'].unique())}")
display(training_data.head())

Training rows: 120861, Features: 159
Label balance: 0.488
Location values: ['A', 'H', 'N']


,team_a_avg_score,team_a_avg_opponent_score,team_a_avg_fgm,team_a_avg_opponent_fgm,team_a_avg_fga,team_a_avg_opponent_fga,team_a_avg_fgm3,team_a_avg_opponent_fgm3,team_a_avg_fga3,team_a_avg_opponent_fga3,...,team_b_col_rank,team_b_ordinal_std,team_b_ordinal_min,season,day_number,team_a_id,team_b_id,label,team_a_location,is_tournament
0,55.0,81.0,20.0,26.0,46.0,57.0,3.0,6.0,11.0,12.0,...,NaN,NaN,NaN,2003,12,1186,1457,1,N,0
1,56.0,50.0,18.0,18.0,38.0,49.0,3.0,6.0,9.0,22.0,...,NaN,NaN,NaN,2003,12,1296,1458,0,A,0
2,48.0,76.0,18.0,25.0,64.0,56.0,8.0,10.0,24.0,23.0,...,NaN,NaN,NaN,2003,14,1125,1135,1,N,0
3,66.0,71.0,24.0,28.0,52.0,58.0,6.0,5.0,18.0,11.0,...,NaN,NaN,NaN,2003,14,1156,1236,1,N,0
4,80.0,62.0,23.0,19.0,55.0,41.0,2.0,4.0,8.0,15.0,...,NaN,NaN,NaN,2003,14,1161,1194,1,H,0


In [8]:
rows = []
for (season, team_id), entry in team_rolling.items():
    row = {'Season': season, 'TeamID': team_id, 'game_count': entry['game_count']}
    for stat_name, total in entry['sums'].items():
        row[f'avg_{stat_name.lower()}'] = total / entry['game_count']
    row['offensive_efficiency'] = entry['offensive_efficiency']
    row['defensive_efficiency'] = entry['defensive_efficiency']
    row['net_efficiency'] = entry['offensive_efficiency'] - entry['defensive_efficiency']
    row['avg_possessions'] = entry['total_possessions'] / entry['game_count']
    rows.append(row)

team_season_stats = pd.DataFrame(rows).sort_values(['Season', 'TeamID']).reset_index(drop=True)

def safe_rate(numerator, denominator):
    """Compute numerator / denominator, returning 0.0 where denominator is zero."""
    return np.where(denominator == 0, 0.0, numerator / denominator)

# Rate stats (from season totals — properly weighted)
team_season_stats['fg_pct'] = safe_rate(team_season_stats['avg_fgm'], team_season_stats['avg_fga'])
team_season_stats['fg3_pct'] = safe_rate(team_season_stats['avg_fgm3'], team_season_stats['avg_fga3'])
team_season_stats['ft_pct'] = safe_rate(team_season_stats['avg_ftm'], team_season_stats['avg_fta'])
team_season_stats['fg3_rate'] = safe_rate(team_season_stats['avg_fga3'], team_season_stats['avg_fga'])

# Same for opponent (defensive) rates
team_season_stats['opponent_fg_pct'] = safe_rate(team_season_stats['avg_opponent_fgm'], team_season_stats['avg_opponent_fga'])
team_season_stats['opponent_fg3_pct'] = safe_rate(team_season_stats['avg_opponent_fgm3'], team_season_stats['avg_opponent_fga3'])
team_season_stats['opponent_ft_pct'] = safe_rate(team_season_stats['avg_opponent_ftm'], team_season_stats['avg_opponent_fta'])
team_season_stats['opponent_fg3_rate'] = safe_rate(team_season_stats['avg_opponent_fga3'], team_season_stats['avg_opponent_fga'])

# Four Factors
team_season_stats['offensive_efg_pct'] = safe_rate(team_season_stats['avg_fgm'] + 0.5 * team_season_stats['avg_fgm3'], team_season_stats['avg_fga'])
team_season_stats['defensive_efg_pct'] = safe_rate(team_season_stats['avg_opponent_fgm'] + 0.5 * team_season_stats['avg_opponent_fgm3'], team_season_stats['avg_opponent_fga'])
team_season_stats['offensive_tov_pct'] = safe_rate(team_season_stats['avg_to'], team_season_stats['avg_fga'] + 0.44 * team_season_stats['avg_fta'] + team_season_stats['avg_to'])
team_season_stats['defensive_tov_pct'] = safe_rate(team_season_stats['avg_opponent_to'], team_season_stats['avg_opponent_fga'] + 0.44 * team_season_stats['avg_opponent_fta'] + team_season_stats['avg_opponent_to'])
team_season_stats['offensive_reb_pct'] = safe_rate(team_season_stats['avg_or'], team_season_stats['avg_or'] + team_season_stats['avg_opponent_dr'])
team_season_stats['defensive_reb_pct'] = safe_rate(team_season_stats['avg_dr'], team_season_stats['avg_opponent_or'] + team_season_stats['avg_dr'])
team_season_stats['offensive_ft_rate'] = safe_rate(team_season_stats['avg_ftm'], team_season_stats['avg_fga'])
team_season_stats['defensive_ft_rate'] = safe_rate(team_season_stats['avg_opponent_ftm'], team_season_stats['avg_opponent_fga'])

display(team_season_stats)

,Season,TeamID,game_count,avg_score,avg_opponent_score,avg_fgm,avg_opponent_fgm,avg_fga,avg_opponent_fga,avg_fgm3,...,opponent_ft_pct,opponent_fg3_rate,offensive_efg_pct,defensive_efg_pct,offensive_tov_pct,defensive_tov_pct,offensive_reb_pct,defensive_reb_pct,offensive_ft_rate,defensive_ft_rate
0,2003,1102,28,57.250000,57.000000,19.142857,19.285714,39.785714,42.428571,7.821429,...,0.710575,0.292929,0.579443,0.510522,0.194557,0.203002,0.171806,0.636486,0.280072,0.322391
1,2003,1103,27,78.777778,78.148148,27.148148,27.777778,55.851852,57.000000,5.444444,...,0.719064,0.322287,0.534814,0.545809,0.158154,0.186813,0.307334,0.623407,0.340849,0.279402
2,2003,1104,29,69.034483,65.068966,23.965517,23.103448,57.000000,55.275862,6.310345,...,0.715415,0.345602,0.475802,0.475671,0.165370,0.178247,0.372249,0.684524,0.259528,0.225827
3,2003,1105,26,71.769231,76.653846,24.384615,27.000000,61.615385,58.961538,7.576923,...,0.668760,0.297456,0.457241,0.511089,0.207538,0.212398,0.338476,0.636653,0.250312,0.277886
4,2003,1106,28,63.607143,63.750000,23.428571,21.714286,55.285714,53.392857,6.107143,...,0.707317,0.284950,0.479005,0.451505,0.214109,0.192905,0.354639,0.678173,0.192506,0.290970
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8341,2026,1477,30,67.066667,76.033333,23.433333,26.833333,55.800000,58.600000,8.066667,...,0.696370,0.379977,0.492234,0.528726,0.164856,0.140514,0.231491,0.685924,0.217443,0.240046
8342,2026,1478,31,72.322581,73.903226,24.709677,25.387097,54.129032,60.258065,7.580645,...,0.737705,0.396681,0.526520,0.492773,0.163887,0.123492,0.229387,0.722846,0.283075,0.240899
8343,2026,1479,32,68.250000,68.656250,25.750000,23.625000,57.906250,52.843750,5.781250,...,0.695833,0.336487,0.494603,0.501478,0.118072,0.158437,0.237762,0.725159,0.189423,0.296274
8344,2026,1480,30,75.333333,80.500000,27.766667,28.400000,62.900000,60.100000,6.866667,...,0.733835,0.373821,0.496025,0.534387,0.130997,0.123767,0.293372,0.742765,0.205617,0.270660


In [9]:
# Verify: top 3 teams by net efficiency, last 3 seasons
recent = (team_season_stats[team_season_stats['Season'] >= 2024]
          .merge(data['MTeams'][['TeamID', 'TeamName']], on='TeamID'))

display(recent.sort_values('net_efficiency', ascending=False)
        .groupby('Season').head(3)
        .sort_values(['Season', 'net_efficiency'], ascending=[True, False])
        [['Season', 'TeamName', 'game_count', 'avg_score', 'offensive_efficiency',
          'defensive_efficiency', 'net_efficiency']])

,Season,TeamName,game_count,avg_score,offensive_efficiency,defensive_efficiency,net_efficiency
56,2024,Connecticut,40,81.400000,125.376147,85.950003,39.426145
113,2024,Houston,37,73.513514,116.175261,85.851816,30.323444
233,2024,Purdue,39,82.333333,120.137588,92.755039,27.382549
436,2025,Duke,39,83.230769,127.331574,90.582431,36.749142
475,2025,Houston,40,73.650000,118.775745,84.498312,34.277433
451,2025,Florida,40,84.775000,126.169297,92.221768,33.947529
800,2026,Duke,34,82.294118,124.650296,88.803729,35.846567
736,2026,Arizona,34,86.147059,124.334545,90.793718,33.540827
815,2026,Florida,33,86.787879,123.341229,91.408456,31.932773


In [14]:
# --- Configuration & feature selection ---
FEATURE_MODE = "concat"  # "diff", "concat", or "all"

# Fill NaN ordinal features (for early-season games before rankings are published)
ORDINAL_RANK_FILL = 176  # neutral midpoint for teams with no ranking yet

ordinal_rank_columns = [col for col in training_data.columns
                        if col.endswith('_rank') and ('ordinal' in col or 'pom' in col or 'mor' in col or 'col' in col)]
ordinal_std_columns = [col for col in training_data.columns if col.endswith('_ordinal_std')]
ordinal_min_columns = [col for col in training_data.columns if col.endswith('_ordinal_min')]

for col in ordinal_rank_columns:
    training_data[col] = training_data[col].fillna(ORDINAL_RANK_FILL)
for col in ordinal_std_columns:
    training_data[col] = training_data[col].fillna(0.0)
for col in ordinal_min_columns:
    training_data[col] = training_data[col].fillna(ORDINAL_RANK_FILL)

print(f"Ordinal rank coverage: {(training_data['team_a_ordinal_rank'] != ORDINAL_RANK_FILL).mean():.1%} of rows have real rankings")
print(f"Filled columns: {sorted(ordinal_rank_columns + ordinal_std_columns + ordinal_min_columns)}")

# Build allowed column set from ALLOWED_FEATURES allowlist
allowed_columns = set()
for base_name in ALLOWED_FEATURES:
    allowed_columns.add(f'team_a_{base_name}')
    allowed_columns.add(f'team_b_{base_name}')
# Ordinals and seeds are always included (not in ALLOWED_FEATURES but matched separately)
ordinal_feature_names = {'ordinal_rank', 'ordinal_std', 'ordinal_min'}
for sys in ORDINAL_INDIVIDUAL_SYSTEMS:
    ordinal_feature_names.add(f'{sys.lower()}_rank')
for base_name in ordinal_feature_names:
    allowed_columns.add(f'team_a_{base_name}')
    allowed_columns.add(f'team_b_{base_name}')
allowed_columns.add('team_a_seed')
allowed_columns.add('team_b_seed')

metadata_columns = {"season", "day_number", "team_a_id", "team_b_id", "label",
                    "team_a_location", "is_tournament"}
feature_columns = [col for col in training_data.columns
                   if col in allowed_columns and col not in metadata_columns]

seed_columns = {"team_a_seed", "team_b_seed"}
feature_columns_no_seed = [col for col in feature_columns if col not in seed_columns]

y = training_data["label"].values
seasons = training_data["season"].values
is_tournament = training_data["is_tournament"].values
sample_weights = np.where(is_tournament == 1, TOURNAMENT_WEIGHT, 1.0)

print(f"Feature mode: {FEATURE_MODE} → {len(feature_columns)} features (incl seed), {len(feature_columns_no_seed)} without seed")
print(f"\nFeature columns:")
for i, col in enumerate(feature_columns, 1):
    print(f"  {i:>3}. {col}")

# --- Cross-validation folds (expanding window by season) ---
VALIDATION_SEASONS = list(range(2016, 2026))

folds = []
for holdout_season in VALIDATION_SEASONS:
    train_mask = seasons < holdout_season
    validation_mask = seasons == holdout_season
    if train_mask.sum() > 0 and validation_mask.sum() > 0:
        folds.append((np.where(train_mask)[0], np.where(validation_mask)[0]))

print(f"\nCross-validation: {len(folds)} folds (seasons {VALIDATION_SEASONS[0]}–{VALIDATION_SEASONS[-1]})")

# --- Model definitions ---
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs"),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=200, max_depth=4, learning_rate=0.1, random_state=42,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42,
        eval_metric="logloss",
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42,
        verbose=-1,
    ),
}

# --- Training loop (per-model feature sets) ---
results_table = []

for model_name, model_template in models.items():
    fold_aucs, fold_briers = [], []
    tournament_aucs, tournament_briers = [], []

    # LR can't handle NaN seed columns; HistGradientBoosting handles them natively
    if model_name == "LogisticRegression":
        model_features = feature_columns_no_seed
        use_scaler = True
    else:
        model_features = feature_columns
        use_scaler = False

    X_model = training_data[model_features].values

    for fold_index, (train_indices, validation_indices) in enumerate(folds):
        if use_scaler:
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_model[train_indices])
            X_validation = scaler.transform(X_model[validation_indices])
        else:
            X_train = X_model[train_indices]
            X_validation = X_model[validation_indices]

        y_train = y[train_indices]
        y_validation = y[validation_indices]

        model = clone(model_template)
        model.fit(X_train, y_train, sample_weight=sample_weights[train_indices])
        probabilities = model.predict_proba(X_validation)[:, 1]

        fold_aucs.append(roc_auc_score(y_validation, probabilities))
        fold_briers.append(brier_score_loss(y_validation, probabilities))

        # Tournament-only metrics for this fold
        tournament_mask = is_tournament[validation_indices] == 1
        if tournament_mask.sum() > 0:
            tournament_aucs.append(roc_auc_score(y_validation[tournament_mask], probabilities[tournament_mask]))
            tournament_briers.append(brier_score_loss(y_validation[tournament_mask], probabilities[tournament_mask]))

    results_table.append({
        "model": model_name,
        "features": len(model_features),
        "auc_mean": np.mean(fold_aucs),
        "auc_std": np.std(fold_aucs),
        "brier_mean": np.mean(fold_briers),
        "brier_std": np.std(fold_briers),
        "tournament_auc_mean": np.mean(tournament_aucs),
        "tournament_auc_std": np.std(tournament_aucs),
        "tournament_brier_mean": np.mean(tournament_briers),
        "tournament_brier_std": np.std(tournament_briers),
    })

# --- Results table ---
print(f"\n{'Model':<22} {'Feats':>5} {'ROC AUC':>18} {'Brier':>18} {'Tourney AUC':>18} {'Tourney Brier':>18}")
print("=" * 102)
for row in results_table:
    print(f"{row['model']:<22} "
          f"{row['features']:>5} "
          f"{row['auc_mean']:.4f} ± {row['auc_std']:.4f}   "
          f"{row['brier_mean']:.4f} ± {row['brier_std']:.4f}   "
          f"{row['tournament_auc_mean']:.4f} ± {row['tournament_auc_std']:.4f}   "
          f"{row['tournament_brier_mean']:.4f} ± {row['tournament_brier_std']:.4f}")

Ordinal rank coverage: 97.3% of rows have real rankings
Filled columns: ['team_a_col_rank', 'team_a_mor_rank', 'team_a_ordinal_min', 'team_a_ordinal_rank', 'team_a_ordinal_std', 'team_a_pom_rank', 'team_b_col_rank', 'team_b_mor_rank', 'team_b_ordinal_min', 'team_b_ordinal_rank', 'team_b_ordinal_std', 'team_b_pom_rank']
Feature mode: concat → 72 features (incl seed), 70 without seed

Feature columns:
    1. team_a_avg_or
    2. team_a_avg_opponent_or
    3. team_a_avg_dr
    4. team_a_avg_opponent_dr
    5. team_a_avg_ast
    6. team_a_avg_opponent_ast
    7. team_a_avg_to
    8. team_a_avg_opponent_to
    9. team_a_avg_stl
   10. team_a_avg_blk
   11. team_a_fg_pct
   12. team_a_fg3_pct
   13. team_a_ft_pct
   14. team_a_fg3_rate
   15. team_a_opponent_fg_pct
   16. team_a_opponent_fg3_pct
   17. team_a_opponent_ft_pct
   18. team_a_opponent_fg3_rate
   19. team_a_offensive_efg_pct
   20. team_a_defensive_efg_pct
   21. team_a_offensive_tov_pct
   22. team_a_defensive_tov_pct
   23. te

In [15]:
# --- Tournament weight sweep ---
weight_candidates = [1, 5, 10, 25, 50, 100]

print(f"{'Weight':<10}", end="")
for model_name in models:
    print(f"  {model_name:>22} Tourney Brier", end="")
print()
print("=" * (10 + 36 * len(models)))

best_weight = {}
best_brier = {}

for weight in weight_candidates:
    sweep_weights = np.where(is_tournament == 1, weight, 1.0)
    print(f"{weight:<10}", end="")

    for model_name, model_template in models.items():
        tournament_briers = []

        # Per-model feature selection (same logic as main CV loop)
        if model_name == "LogisticRegression":
            model_features = feature_columns_no_seed
            use_scaler = True
        else:
            model_features = feature_columns
            use_scaler = False

        X_model = training_data[model_features].values

        for train_indices, validation_indices in folds:
            if use_scaler:
                scaler = StandardScaler()
                X_train = scaler.fit_transform(X_model[train_indices])
                X_validation = scaler.transform(X_model[validation_indices])
            else:
                X_train = X_model[train_indices]
                X_validation = X_model[validation_indices]

            y_train = y[train_indices]
            y_validation = y[validation_indices]

            model = clone(model_template)
            model.fit(X_train, y_train, sample_weight=sweep_weights[train_indices])
            probabilities = model.predict_proba(X_validation)[:, 1]

            tournament_mask = is_tournament[validation_indices] == 1
            if tournament_mask.sum() > 0:
                tournament_briers.append(brier_score_loss(y_validation[tournament_mask], probabilities[tournament_mask]))

        mean_brier = np.mean(tournament_briers)
        print(f"  {mean_brier:>35.4f}", end="")

        if model_name not in best_brier or mean_brier < best_brier[model_name]:
            best_brier[model_name] = mean_brier
            best_weight[model_name] = weight

    print()

print()
for model_name in models:
    print(f"Best weight for {model_name}: {best_weight[model_name]} (tourney Brier: {best_brier[model_name]:.4f})")

Weight          LogisticRegression Tourney Brier    HistGradientBoosting Tourney Brier                 XGBoost Tourney Brier                LightGBM Tourney Brier
1                                        0.1934                               0.1911                               0.1913                               0.1908
5                                        0.1931                               0.1966                               0.1948                               0.1916
10                                       0.1930                               0.1936                               0.1986                               0.1992
25                                       0.1926                               0.1936                               0.2076                               0.1995
50                                       0.1925                               0.1934                               0.2028                               0.2034
100                                      0

In [ ]:
# --- Train final model (LightGBM on all data) ---
final_model = LGBMClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42, verbosity=-1,
)
X_all = training_data[feature_columns].values
y_all = training_data['label'].values
sample_weights_all = np.where(training_data['is_tournament'].values == 1, TOURNAMENT_WEIGHT, 1.0)
final_model.fit(X_all, y_all, sample_weight=sample_weights_all)
print(f"Final model: LightGBM trained on {len(y_all)} games with {len(feature_columns)} features")
print(f"Tournament weight: {TOURNAMENT_WEIGHT}")

In [ ]:
# --- 2026 First Round Predictions (Sanity Check) ---

# 1. Build lookups for 2026
seed_to_team = {}
for _, row in data['MNCAATourneySeeds'][data['MNCAATourneySeeds']['Season'] == 2026].iterrows():
    seed_to_team[row['Seed']] = row['TeamID']

team_names = dict(zip(data['MTeams']['TeamID'], data['MTeams']['TeamName']))

TOURNAMENT_DAY = 136  # approximate first round day

def predict_matchup(team_id_1, team_id_2, seed_num_1, seed_num_2):
    """Predict win probability for team_id_1 using bidirectional averaging.

    Featurizes the game both ways (lower ID as team_a, then higher ID as team_a)
    and averages to reduce ordering bias.

    Returns probability that team_id_1 wins.
    """
    entry_1 = team_rolling[(2026, team_id_1)]
    entry_2 = team_rolling[(2026, team_id_2)]
    ordinals_1 = build_ordinal_features(2026, team_id_1, TOURNAMENT_DAY)
    ordinals_2 = build_ordinal_features(2026, team_id_2, TOURNAMENT_DAY)

    lower_id = min(team_id_1, team_id_2)
    higher_id = max(team_id_1, team_id_2)

    if team_id_1 < team_id_2:
        lower_entry, higher_entry = entry_1, entry_2
        lower_seed, higher_seed = seed_num_1, seed_num_2
        lower_ordinals, higher_ordinals = ordinals_1, ordinals_2
    else:
        lower_entry, higher_entry = entry_2, entry_1
        lower_seed, higher_seed = seed_num_2, seed_num_1
        lower_ordinals, higher_ordinals = ordinals_2, ordinals_1

    # Way 1: lower_id as team_a (standard orientation)
    row_forward = featurize_game(
        lower_entry, higher_entry, 2026, TOURNAMENT_DAY,
        lower_id, higher_id, label=0, team_a_location='N',
        is_tournament=1, team_a_seed=lower_seed, team_b_seed=higher_seed,
        team_a_ordinals=lower_ordinals, team_b_ordinals=higher_ordinals)

    # Way 2: higher_id as team_a (reversed orientation)
    row_reversed = featurize_game(
        higher_entry, lower_entry, 2026, TOURNAMENT_DAY,
        higher_id, lower_id, label=0, team_a_location='N',
        is_tournament=1, team_a_seed=higher_seed, team_b_seed=lower_seed,
        team_a_ordinals=higher_ordinals, team_b_ordinals=lower_ordinals)

    # Fill NaN ordinals to match training data preprocessing
    prediction_rows = pd.DataFrame([row_forward, row_reversed])
    for col in ordinal_rank_columns:
        prediction_rows[col] = prediction_rows[col].fillna(ORDINAL_RANK_FILL)
    for col in ordinal_std_columns:
        prediction_rows[col] = prediction_rows[col].fillna(0.0)
    for col in ordinal_min_columns:
        prediction_rows[col] = prediction_rows[col].fillna(ORDINAL_RANK_FILL)

    X_forward = prediction_rows.iloc[[0]][feature_columns].values
    X_reversed = prediction_rows.iloc[[1]][feature_columns].values

    prob_lower_wins_forward = final_model.predict_proba(X_forward)[0, 1]
    prob_higher_wins_reversed = final_model.predict_proba(X_reversed)[0, 1]

    # Average the two orientations: P(lower wins) from both perspectives
    prob_lower_wins = (prob_lower_wins_forward + (1 - prob_higher_wins_reversed)) / 2

    if team_id_1 == lower_id:
        return prob_lower_wins
    else:
        return 1 - prob_lower_wins

# 2. Resolve play-in games
slots_2026 = data['MNCAATourneySlots'][data['MNCAATourneySlots']['Season'] == 2026]
playin_slots = slots_2026[~slots_2026['Slot'].str.startswith('R')]
first_round_slots = slots_2026[slots_2026['Slot'].str.startswith('R1')]

print("\n=== PLAY-IN GAMES ===\n")
playin_winners = {}
for _, slot in playin_slots.iterrows():
    strong_team = seed_to_team[slot['StrongSeed']]
    weak_team = seed_to_team[slot['WeakSeed']]
    strong_seed_num = seed_lookup[(2026, strong_team)]
    weak_seed_num = seed_lookup[(2026, weak_team)]

    prob_strong_wins = predict_matchup(strong_team, weak_team, strong_seed_num, weak_seed_num)
    winner_id = strong_team if prob_strong_wins > 0.5 else weak_team
    playin_winners[slot['Slot']] = winner_id

    marker = "←" if prob_strong_wins > 0.5 else " "
    print(f"  {slot['Slot']:4s}: ({strong_seed_num:>2d}) {team_names[strong_team]:<22s} {prob_strong_wins:5.1%}  {marker}")
    marker = " " if prob_strong_wins > 0.5 else "←"
    print(f"        ({weak_seed_num:>2d}) {team_names[weak_team]:<22s} {1 - prob_strong_wins:5.1%}  {marker}")
    print()

# 3. Predict all 32 first round games
region_labels = {'W': 'WEST', 'X': 'EAST', 'Y': 'SOUTH', 'Z': 'MIDWEST'}
predictions = []

for _, slot in first_round_slots.iterrows():
    region = slot['Slot'][2]  # R1W1 -> 'W'

    # Resolve seeds (play-in winners fill their parent slot)
    strong_seed_str = slot['StrongSeed']
    strong_team = playin_winners.get(strong_seed_str, seed_to_team.get(strong_seed_str))

    weak_seed_str = slot['WeakSeed']
    weak_team = playin_winners.get(weak_seed_str, seed_to_team.get(weak_seed_str))

    strong_seed_num = seed_lookup[(2026, strong_team)]
    weak_seed_num = seed_lookup[(2026, weak_team)]

    prob_strong_wins = predict_matchup(strong_team, weak_team, strong_seed_num, weak_seed_num)

    predictions.append({
        'region': region,
        'strong_seed': strong_seed_num,
        'weak_seed': weak_seed_num,
        'strong_team': team_names[strong_team],
        'weak_team': team_names[weak_team],
        'strong_seed_win_prob': prob_strong_wins,
    })

# 4. Display grouped by region
print("=== FIRST ROUND PREDICTIONS ===\n")
for region_code in ['W', 'X', 'Y', 'Z']:
    region_preds = [p for p in predictions if p['region'] == region_code]
    region_preds.sort(key=lambda p: p['strong_seed'])
    print(f"--- {region_labels[region_code]} ---")
    print(f"  {'Matchup':<52s} {'Fav %':>5s}")
    print(f"  {'─' * 58}")
    for p in region_preds:
        matchup = f"({p['strong_seed']:>2d}) {p['strong_team']:<22s} vs ({p['weak_seed']:>2d}) {p['weak_team']}"
        print(f"  {matchup:<52s} {p['strong_seed_win_prob']:5.1%}")
    print()